In [1]:
import os
from pathlib import Path

In [ ]:
path = Path("../data/responses_1/")
institution_ids = [os.path.basename(p).replace(".json", "") for p in  path.rglob('*.json')]
institution_ids = set(institution_ids)
print(len(institution_ids))

1680565


In [3]:
import json
citation_index = json.load(open("../data/output/citation_index.json"))

In [ ]:
print(f"Number of unique nodes in the citation network: {len(set(citation_index.keys()))}")

total_edges = 0
edges_with_downloaded_ids = 0

for node, info in citation_index.items():
    index = info.get("index", [])
    reverse_index = info.get("reverse_index", [])
    if index:
        total_edges += len(index)
        edges_with_downloaded_ids += sum(1 for cited in index if cited in institution_ids)
    del index
    del reverse_index

Number of unique nodes in the citation network: 8953263


In [9]:
print(f"Number of edges in the citation network: {total_edges}")
print(f"Number of edges containing ids in the ID set: {edges_with_downloaded_ids}")

Number of edges in the citation network: 47991643
Number of edges containing ids in the ID set: 22123799


In [ ]:
print(f"Number of unique nodes in the citation network and in the ID set: {len(set(citation_index.keys()) & institution_ids)}")

Number of unique nodes in the citation network and in the ID set: 1680563


In [ ]:
path = Path("../data/responses_1/")
institution_ids = [os.path.basename(p).replace(".json", "") for p in  path.rglob('*.json')]
institution_ids = set(institution_ids)
print(len(institution_ids))

In [5]:
institution_paths = [
    Path("../data/responses_1/by_institution/universidade_federal_rio_janei"),
    Path("../data/responses_1/by_institution/massachusetts_institute_techno"),
    Path("../data/responses_1/by_institution/universidade_federal_sao_carlo"),
    Path("../data/responses_1/by_institution/carnegie_mellon_university"),
    Path("../data/responses_1/by_institution/universidade_sao_paulo"),
    Path("../data/responses_1/by_institution/nanjing_university"),
    Path("../data/responses_1/by_institution/university_california_berkeley"),
    Path("../data/responses_1/by_institution/eindhoven_university_technolog"),
    Path("../data/responses_1/by_institution/national_university_singapore"),
    Path("../data/responses_1/by_institution/federal_university_minas_gerai"),
    Path("../data/responses_1/by_institution/harvard_university"),
    Path("../data/responses_1/by_institution/peking_university"),
    Path("../data/responses_1/by_institution/university_melbourne"),
    Path("../data/responses_1/by_institution/humboldt-universitat_berlin"),
    Path("../data/responses_1/by_institution/university_oxford"),
    Path("../data/responses_1/by_institution/university_pennsylvania"),
    Path("../data/responses_1/by_institution/stanford_university"),
    Path("../data/responses_1/by_institution/university_tokyo"),
    Path("../data/responses_1/by_institution/instituto_tecnologico_aeronaut"),
    Path("../data/responses_1/by_institution/technical_university_darmstadt"),
    Path("../data/responses_1/by_institution/institut_polytechnique_paris"),
    Path("../data/responses_1/by_institution/technical_university_munich"),
    Path("../data/responses_1/by_institution/vellore_institute_technology_v"),
    Path("../data/responses_1/by_institution/wuhan_university"),
    Path("../data/responses_1/by_institution/king_s_college_london"),
    Path("../data/responses_1/by_institution/tsinghua_university"),
    Path("../data/responses_1/by_institution/yale_university"),
    Path("../data/responses_1/by_institution/kth_royal_institute_technology"),
    Path("../data/responses_1/by_institution/unesp"),
    Path("../data/responses_1/by_institution/kyoto_university")
]

In [6]:
institution_ids = set()
for path in institution_paths:
    institution_ids.update([os.path.basename(p).replace(".json", "") for p in path.rglob('*.json')])

In [11]:
print(f"Number of unique institution ids: {len(institution_ids)}")

Number of unique institution ids: 153464


In [23]:
unicamp_path = Path("../data/responses_1/_unicamp_cs")
unicamp_ids = set(os.path.basename(p).replace(".json", "") for p in unicamp_path.rglob('*.json'))
print(f"Number of unique institution ids in unicamp: {len(unicamp_ids)}")

top_tier_path = Path("../data/responses_1/_top_cited_cs")
top_tier_ids = set(os.path.basename(p).replace(".json", "") for p in top_tier_path.rglob('*.json'))
print(f"Number of unique institution ids in top_tier: {len(top_tier_ids)}")

Number of unique institution ids in unicamp: 2018
Number of unique institution ids in top_tier: 461


In [15]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [19]:
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

In [20]:
from neo4j import GraphDatabase
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
all_articles_query = """
MATCH (a:Article)
RETURN a.openalex_id
"""
with neo4j_driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(all_articles_query)
    article_ids_in_neo4j = set(record["a.openalex_id"] for record in result)
print(f"Number of unique article nodes in Neo4j: {len(article_ids_in_neo4j)}")

Number of unique article nodes in Neo4j: 49196


In [ ]:
all_citations_query = """
MATCH (a:Article)-[:CITES]->(b:Article)
RETURN a.openalex_id AS citing, b.openalex_id AS cited
"""
with neo4j_driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(all_citations_query)
    citations_in_neo4j = set((record["citing"], record["cited"]) for record in result)
print(f"Number of unique citation edges in Neo4j: {len(citations_in_neo4j)}")

In [24]:
print(f"Number of articles from institutions in the ID set that are present in Neo4j: {len(article_ids_in_neo4j & institution_ids)}")
print(f"Number of articles from Unicamp that are present in Neo4j: {len(article_ids_in_neo4j & unicamp_ids)}")
print(f"Number of articles from top tier that are present in Neo4j: {len(article_ids_in_neo4j & top_tier_ids)}")

Number of articles from institutions in the ID set that are present in Neo4j: 6207
Number of articles from Unicamp that are present in Neo4j: 1907
Number of articles from top tier that are present in Neo4j: 298
